# Single Notebook CNN Investigation (iCoSimal V3)

This notebook is organized directly by the **mandatory objectives (a–e)** and includes **initial data analysis**.

## Mandatory objectives checklist
- **(a)** Start from simple CNN architectures and progressively increase complexity/depth.
- **(b)** Show the importance of hyperparameter tuning.
- **(c)** Show underfitting and overfitting cases and explain reasons.
- **(d)** Experiment with regularization techniques.
- **(e)** Compare optimization algorithms (Adam, RMSprop, SGD).

> Practical tip: begin with smaller image sizes (64×64 / 128×128) for quicker experimentation.


## 0) Setup


In [ ]:
# If needed, install dependencies first:
# !pip install -r ../requirements.txt

import os
import sys
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

# Notebook -> repository root
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.data_loader import get_dataloaders, get_transforms
from src.models import get_model, count_parameters
from src.train import train
from src.evaluate import (
    predict,
    print_classification_report,
    plot_confusion_matrix,
    plot_training_history,
    compare_architectures,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
# Set this to your local dataset path (must contain train/ and validate/)
DATA_ROOT = '/path/to/icosimal_img_class_03/data_uniform_224_224_sets'

# Global quick-run defaults (increase later for stronger experiments)
DEFAULT_IMAGE_SIZE = 64
DEFAULT_BATCH_SIZE = 32
DEFAULT_EPOCHS = 8
NUM_WORKERS = 4


## 1) Initial Data Analysis


In [ ]:
# Basic dataset existence checks
train_dir = Path(DATA_ROOT) / 'train'
val_dir = Path(DATA_ROOT) / 'validate'
print('Train dir:', train_dir)
print('Val dir:  ', val_dir)
print('Train exists?', train_dir.exists())
print('Val exists?  ', val_dir.exists())

if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError('Please set DATA_ROOT to the extracted dataset folder with train/ and validate/.')


In [ ]:
# Class counts and split summary
class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])

train_counts = {c: len(list((train_dir / c).glob('*'))) for c in class_names}
val_counts = {c: len(list((val_dir / c).glob('*'))) for c in class_names}

eda_df = pd.DataFrame({
    'class': class_names,
    'train_count': [train_counts[c] for c in class_names],
    'val_count': [val_counts[c] for c in class_names],
})
eda_df['total'] = eda_df['train_count'] + eda_df['val_count']

display(eda_df)
print('Total train images:', eda_df['train_count'].sum())
print('Total val images:  ', eda_df['val_count'].sum())


In [ ]:
# Plot class distribution for train/validation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=eda_df, x='class', y='train_count', ax=axes[0], color='steelblue')
axes[0].set_title('Train split class counts')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=eda_df, x='class', y='val_count', ax=axes[1], color='darkorange')
axes[1].set_title('Validation split class counts')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Sample raw image-size statistics (before transforms)
def sample_image_sizes(root_dir, max_per_class=200):
    sizes = []
    for cls_dir in sorted([p for p in Path(root_dir).iterdir() if p.is_dir()]):
        for i, img_path in enumerate(cls_dir.glob('*')):
            if i >= max_per_class:
                break
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
                sizes.append((cls_dir.name, w, h))
            except Exception:
                continue
    return pd.DataFrame(sizes, columns=['class', 'width', 'height'])

size_df = sample_image_sizes(train_dir)
display(size_df.describe(include='all'))
\n

In [ ]:
# Visualize a mini-batch after transforms
train_loader, val_loader, CLASS_NAMES = get_dataloaders(
    data_root=DATA_ROOT,
    image_size=DEFAULT_IMAGE_SIZE,
    batch_size=DEFAULT_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment=True,
)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    img = images[i].permute(1, 2, 0).cpu().numpy()
    # de-normalize roughly for display
    img = np.clip((img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406]), 0, 1)
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[i].item()])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 2) Objective (a): Progressive Depth (Simple → Medium → Deep)

We keep training settings fixed and only change architecture depth/complexity.


In [ ]:
DEPTH_CFG = dict(
    image_size=64,
    batch_size=32,
    num_epochs=8,
    learning_rate=1e-3,
    optimizer_name='adam',
    weight_decay=1e-4,
    dropout=0.5,
    scheduler_name='cosine',
    augment=True,
)

train_loader, val_loader, CLASS_NAMES = get_dataloaders(
    DATA_ROOT,
    image_size=DEPTH_CFG['image_size'],
    batch_size=DEPTH_CFG['batch_size'],
    num_workers=NUM_WORKERS,
    augment=DEPTH_CFG['augment'],
)

arch_histories = {}
arch_models = {}
for arch in ['simple', 'medium', 'deep']:
    print(f"\n=== Training architecture: {arch.upper()} ===")
    model = get_model(
        architecture=arch,
        num_classes=len(CLASS_NAMES),
        input_size=DEPTH_CFG['image_size'],
        dropout=DEPTH_CFG['dropout'],
    )
    print('Trainable params:', f"{count_parameters(model):,}")

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=DEPTH_CFG['num_epochs'],
        learning_rate=DEPTH_CFG['learning_rate'],
        optimizer_name=DEPTH_CFG['optimizer_name'],
        weight_decay=DEPTH_CFG['weight_decay'],
        scheduler_name=DEPTH_CFG['scheduler_name'],
        device=DEVICE,
        verbose=True,
    )
    arch_histories[arch] = history
    arch_models[arch] = model


In [ ]:
# Compare validation accuracy curves for objective (a)
_ = compare_architectures(arch_histories, metric='val_acc', figsize=(9, 5))
plt.show()

depth_summary = pd.DataFrame([
    {
        'architecture': arch,
        'best_val_acc': max(hist['val_acc']),
        'final_train_acc': hist['train_acc'][-1],
        'params': count_parameters(arch_models[arch]),
    }
    for arch, hist in arch_histories.items()
]).sort_values('best_val_acc', ascending=False)

display(depth_summary)


### Objective (a) Interpretation
- As depth increases, the model has greater representational power and usually reaches higher validation accuracy.
- The deep model also benefits from BatchNorm and a stronger feature hierarchy.
- If deeper models do not outperform shallower ones, likely causes are insufficient epochs, overly small learning rate, or regularization settings.


## 3) Objective (b): Hyperparameter Tuning Importance

We run a compact sweep first (fast) and identify which settings move validation accuracy the most.


In [ ]:
hparam_grid = {
    'image_size': [64, 128],
    'batch_size': [32, 64],
    'learning_rate': [1e-3, 3e-4, 1e-4],
    'optimizer_name': ['adam'],
    'weight_decay': [1e-4],
    'dropout': [0.3, 0.5],
    'scheduler_name': ['cosine'],
    'architecture': ['medium'],
    'num_epochs': [6],
}

keys = list(hparam_grid.keys())
configs = [dict(zip(keys, vals)) for vals in product(*[hparam_grid[k] for k in keys])]
print('Total sweep configurations:', len(configs))

sweep_records = []
for i, cfg in enumerate(configs, 1):
    print(f"\n[{i}/{len(configs)}] {cfg}")
    train_loader, val_loader, _ = get_dataloaders(
        DATA_ROOT,
        image_size=cfg['image_size'],
        batch_size=cfg['batch_size'],
        num_workers=NUM_WORKERS,
        augment=True,
    )
    model = get_model(
        architecture=cfg['architecture'],
        num_classes=10,
        input_size=cfg['image_size'],
        dropout=cfg['dropout'],
    )
    history = train(
        model,
        train_loader,
        val_loader,
        num_epochs=cfg['num_epochs'],
        learning_rate=cfg['learning_rate'],
        optimizer_name=cfg['optimizer_name'],
        weight_decay=cfg['weight_decay'],
        scheduler_name=cfg['scheduler_name'],
        device=DEVICE,
        verbose=False,
    )
    sweep_records.append({
        **cfg,
        'best_val_acc': max(history['val_acc']),
        'final_val_acc': history['val_acc'][-1],
        'final_train_acc': history['train_acc'][-1],
    })

sweep_df = pd.DataFrame(sweep_records).sort_values('best_val_acc', ascending=False)
display(sweep_df.head(10))


In [ ]:
# Visualize tuning effects
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=sweep_df, x='image_size', y='best_val_acc', ax=axes[0])
axes[0].set_title('Impact of image_size')

sns.boxplot(data=sweep_df, x='batch_size', y='best_val_acc', ax=axes[1])
axes[1].set_title('Impact of batch_size')

lr_effect = sweep_df.groupby('learning_rate', as_index=False)['best_val_acc'].mean().sort_values('learning_rate')
axes[2].plot(lr_effect['learning_rate'], lr_effect['best_val_acc'], marker='o')
axes[2].set_xscale('log')
axes[2].set_title('Impact of learning_rate (mean best val acc, log scale)')

plt.tight_layout()
plt.show()
\n

### Objective (b) Interpretation
- Small hyperparameter choices (especially learning rate and image size) can noticeably change validation performance.
- Tuning can produce bigger gains than architecture swaps under fixed training budgets.
- Start with small image size and fewer epochs for fast iteration, then scale up promising configurations.


## 4) Objective (c): Underfitting vs Overfitting

We intentionally configure one model to underfit and one to overfit, then compare learning curves.


In [ ]:
# --- Underfitting setup ---
underfit_train_loader, underfit_val_loader, _ = get_dataloaders(
    DATA_ROOT,
    image_size=64,
    batch_size=64,
    num_workers=NUM_WORKERS,
    augment=True,
)
underfit_model = get_model('simple', num_classes=10, input_size=64, dropout=0.7)
underfit_history = train(
    underfit_model,
    underfit_train_loader,
    underfit_val_loader,
    num_epochs=4,
    learning_rate=1e-4,
    optimizer_name='sgd',
    weight_decay=1e-2,
    scheduler_name='none',
    device=DEVICE,
    verbose=True,
)

# --- Overfitting setup (small subset + weak regularization) ---
train_tf, val_tf = get_transforms(image_size=64, augment=False)
full_train_dataset = datasets.ImageFolder(str(train_dir), transform=train_tf)
small_indices = list(range(min(1000, len(full_train_dataset))))
small_train_dataset = Subset(full_train_dataset, small_indices)

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
_, overfit_val_loader, _ = get_dataloaders(
    DATA_ROOT,
    image_size=64,
    batch_size=32,
    num_workers=NUM_WORKERS,
    augment=False,
)
overfit_model = get_model('deep', num_classes=10, input_size=64, dropout=0.0)
overfit_history = train(
    overfit_model,
    small_train_loader,
    overfit_val_loader,
    num_epochs=12,
    learning_rate=1e-3,
    optimizer_name='adam',
    weight_decay=0.0,
    scheduler_name='none',
    device=DEVICE,
    verbose=True,
)


In [ ]:
# Plot underfit vs overfit histories
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Underfit
axes[0, 0].plot(underfit_history['train_loss'], label='train')
axes[0, 0].plot(underfit_history['val_loss'], label='val')
axes[0, 0].set_title('Underfitting: Loss')
axes[0, 0].legend()

axes[0, 1].plot(underfit_history['train_acc'], label='train')
axes[0, 1].plot(underfit_history['val_acc'], label='val')
axes[0, 1].set_title('Underfitting: Accuracy')
axes[0, 1].legend()

# Overfit
axes[1, 0].plot(overfit_history['train_loss'], label='train')
axes[1, 0].plot(overfit_history['val_loss'], label='val')
axes[1, 0].set_title('Overfitting: Loss')
axes[1, 0].legend()

axes[1, 1].plot(overfit_history['train_acc'], label='train')
axes[1, 1].plot(overfit_history['val_acc'], label='val')
axes[1, 1].set_title('Overfitting: Accuracy')
axes[1, 1].legend()

plt.tight_layout()
plt.show()


### Objective (c) Explanation
- **Underfitting** appears when both train and validation accuracy remain low (model too simple, too few epochs, too strong regularization, or too low LR).
- **Overfitting** appears when train accuracy becomes very high but validation lags or degrades (model too flexible relative to effective data, weak regularization, or very small training subset).


## 5) Objective (d): Regularization Techniques

We compare settings with weaker vs stronger regularization: dropout, weight decay, and augmentation.


In [ ]:
regularization_experiments = [
    {
        'name': 'weak_reg',
        'dropout': 0.0,
        'weight_decay': 0.0,
        'augment': False,
    },
    {
        'name': 'dropout_only',
        'dropout': 0.5,
        'weight_decay': 0.0,
        'augment': False,
    },
    {
        'name': 'dropout_wd_aug',
        'dropout': 0.5,
        'weight_decay': 1e-4,
        'augment': True,
    },
]

reg_histories = {}
reg_summary = []
for cfg in regularization_experiments:
    print(f"\n=== Regularization experiment: {cfg['name']} ===")
    tr_loader, va_loader, _ = get_dataloaders(
        DATA_ROOT,
        image_size=64,
        batch_size=32,
        num_workers=NUM_WORKERS,
        augment=cfg['augment'],
    )
    model = get_model('medium', num_classes=10, input_size=64, dropout=cfg['dropout'])
    history = train(
        model,
        tr_loader,
        va_loader,
        num_epochs=8,
        learning_rate=1e-3,
        optimizer_name='adam',
        weight_decay=cfg['weight_decay'],
        scheduler_name='cosine',
        device=DEVICE,
        verbose=False,
    )
    reg_histories[cfg['name']] = history
    reg_summary.append({
        **cfg,
        'best_val_acc': max(history['val_acc']),
        'final_train_acc': history['train_acc'][-1],
    })

reg_df = pd.DataFrame(reg_summary).sort_values('best_val_acc', ascending=False)
display(reg_df)
_ = compare_architectures(reg_histories, metric='val_acc', figsize=(9, 5))
plt.show()


### Objective (d) Interpretation
- Regularization generally improves generalization by reducing the train/validation gap.
- Dropout and weight decay constrain model co-adaptation.
- Data augmentation acts as data-space regularization and often gives robust gains.


## 6) Objective (e): Optimizer Comparison (Adam vs RMSprop vs SGD)

We keep architecture and most hyperparameters fixed, then compare optimizers.


In [ ]:
optimizer_histories = {}
optimizer_summary = []

for opt_name in ['adam', 'rmsprop', 'sgd']:
    print(f"\n=== Optimizer: {opt_name.upper()} ===")
    tr_loader, va_loader, _ = get_dataloaders(
        DATA_ROOT,
        image_size=64,
        batch_size=32,
        num_workers=NUM_WORKERS,
        augment=True,
    )
    model = get_model('deep', num_classes=10, input_size=64, dropout=0.5)
    history = train(
        model,
        tr_loader,
        va_loader,
        num_epochs=8,
        learning_rate=1e-3 if opt_name != 'sgd' else 3e-2,
        optimizer_name=opt_name,
        weight_decay=1e-4,
        scheduler_name='cosine',
        device=DEVICE,
        verbose=False,
    )
    optimizer_histories[opt_name] = history
    optimizer_summary.append({
        'optimizer': opt_name,
        'best_val_acc': max(history['val_acc']),
        'final_val_acc': history['val_acc'][-1],
        'final_train_acc': history['train_acc'][-1],
    })

opt_df = pd.DataFrame(optimizer_summary).sort_values('best_val_acc', ascending=False)
display(opt_df)
_ = compare_architectures(optimizer_histories, metric='val_acc', figsize=(9, 5))
plt.show()


### Objective (e) Interpretation
- Adam and RMSprop often converge faster in early epochs.
- SGD can match/beat adaptive methods with tuned learning rate and longer training.
- Optimizer performance is data/model dependent, so direct experiment-based comparison is essential.


## 7) Final Evaluation (Best Practical Configuration)

Use the best configuration from the experiments above, retrain with more epochs and (optionally) larger image size (128 or 224), then evaluate with confusion matrix and class report.


In [ ]:
# Example final configuration (replace with your best found settings)
FINAL_CFG = {
    'architecture': 'deep',
    'image_size': 128,
    'batch_size': 32,
    'dropout': 0.5,
    'num_epochs': 15,
    'learning_rate': 1e-3,
    'optimizer_name': 'adam',
    'weight_decay': 1e-4,
    'scheduler_name': 'cosine',
    'augment': True,
}

final_train_loader, final_val_loader, CLASS_NAMES = get_dataloaders(
    DATA_ROOT,
    image_size=FINAL_CFG['image_size'],
    batch_size=FINAL_CFG['batch_size'],
    num_workers=NUM_WORKERS,
    augment=FINAL_CFG['augment'],
)

final_model = get_model(
    FINAL_CFG['architecture'],
    num_classes=len(CLASS_NAMES),
    input_size=FINAL_CFG['image_size'],
    dropout=FINAL_CFG['dropout'],
)

final_history = train(
    final_model,
    final_train_loader,
    final_val_loader,
    num_epochs=FINAL_CFG['num_epochs'],
    learning_rate=FINAL_CFG['learning_rate'],
    optimizer_name=FINAL_CFG['optimizer_name'],
    weight_decay=FINAL_CFG['weight_decay'],
    scheduler_name=FINAL_CFG['scheduler_name'],
    device=DEVICE,
    verbose=True,
)

_ = plot_training_history(final_history, title='Final Model Training History')
plt.show()


In [ ]:
# Classification metrics and confusion matrix on validation split
y_true, y_pred = predict(final_model, final_val_loader, DEVICE)
print_classification_report(y_true, y_pred, CLASS_NAMES)
_ = plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, title='Final Model Confusion Matrix')
plt.show()


In [ ]:
# Save final model
output_dir = REPO_ROOT / 'results'
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / 'best_cnn_model.pth'
torch.save(final_model.state_dict(), model_path)
print('Saved model to:', model_path)


## 8) Consolidated Objective Coverage Summary

- **(a) Progressive depth**: Section 2 compares Simple/Medium/Deep under fixed settings.
- **(b) Hyperparameter tuning**: Section 3 sweeps image size, batch size, learning rate, and dropout.
- **(c) Underfitting/overfitting**: Section 4 intentionally demonstrates and explains both failure modes.
- **(d) Regularization**: Section 5 compares dropout, weight decay, and augmentation combinations.
- **(e) Optimizers**: Section 6 compares Adam, RMSprop, and SGD using the same model backbone.

This single notebook contains the full investigation workflow end-to-end.
